## Filtering transactions

Filtering transactions by removing transactions that already contain a known verb pattern (existing in vp_data3) and modifying the resulting transactions further by removing specific syntactic relations. As an extra step, filtered transactions will be further divided into transactions containing a match to an existing negation pattern (in negations database) and transactions not containing a match to a negation pattern.

In [1]:
import sys
sys.path.append('../../../common_code')

In [2]:
import sqlite3
from paths import PATH_ROOT
from db_operations.index_operations.index_operations import *
from db_operations.verb_transactions.filter_verb_transaction_tables import *
from db_operations.verb_transactions.transaction_modifiers import *
from db_operations.db_display import *

## Input parameters

In [3]:
INPUT_DIR = "C:/Users/liivas/Documents/Töö/verbirektisoonid"

VERB_PATTERNS_DB = f"{INPUT_DIR}/vp_data3.db"
NEG_TABLES_DB = f"{INPUT_DIR}/db_operations/workflows/002_verb_negations/development/neg_tables.db"
TRANSACTION_DB = f"{INPUT_DIR}/v32_data.db"
MATCHED_TR_DB = "matched_transactions.db"
NEG_TR_DB = "negative_transactions.db" # negated transactions
POS_TR_DB = "positive_transactions.db" # positive (non-negated) transactions

## Data processing

In [4]:
con = sqlite3.connect(VERB_PATTERNS_DB)
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS transactions')
cur.execute(f'ATTACH DATABASE "{MATCHED_TR_DB}" AS matched_tr')

# matched transactions
index_difference(cur,
                 index_tbl_1='verb_matches', # verbs matching verbs in vp_data3 patterns
                 id_col_1='head_id',
                 index_tbl_2='verb_phrase_matches', # phrases containing full match to a pattern in vp_data3
                 id_col_2='head_id',
                 output_tbl='matched_tr.index_tbl')

filter_verb_transaction_tables(
    conn=con,
    source_schema='transactions',
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    target_schema='matched_tr',
    new_transaction_head='transaction_head',
    new_transaction_row='transaction_row',
    ids_schema='matched_tr',
    ids_table='index_tbl',
    ids_column='idx',
    delete_if_exists= True,
    copy_indexes=True,
    verbose= True,
)

remove_deprel_from_transaction_row(cur,
                                   transaction_row='matched_tr.transaction_row',
                                   deprel='nsubj')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='matched_tr.transaction_row',
                                   deprel='advmod')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='matched_tr.transaction_row',
                                   deprel='advcl')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='matched_tr.transaction_row',
                                   deprel='csubj')
remove_aux_verbs(cur,
                 transaction_row='matched_tr.transaction_row')

CREATE TABLE "matched_tr"."transaction_head" (`id` INTEGER PRIMARY KEY AUTOINCREMENT, `sentence_id` int, `loc` int, `verb` text, `verb_compound` text, `form` text, `deprel` text, `feats` text)
Created table 'matched_tr.transaction_head' (foreign keys ignored).
Copying indexes from 'transactions.transaction_head' to 'matched_tr.transaction_head'
CREATE INDEX "matched_tr"."0abd3003_`transaction_head_deprel`" ON "transaction_head"("`deprel`" ASC)
Index '0abd3003_`transaction_head_deprel`' created successfully.
CREATE INDEX "matched_tr"."0abd3003_`transaction_head_feats`" ON "transaction_head"("`feats`" ASC)
Index '0abd3003_`transaction_head_feats`' created successfully.
CREATE INDEX "matched_tr"."0abd3003_`transaction_head_verb_compound`" ON "transaction_head"("`verb_compound`" ASC)
Index '0abd3003_`transaction_head_verb_compound`' created successfully.
CREATE INDEX "matched_tr"."0abd3003_`transaction_head_verb`" ON "transaction_head"("`verb`" ASC)
Index '0abd3003_`transaction_head_verb`'

In [6]:
con = sqlite3.connect(VERB_PATTERNS_DB)
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{NEG_TABLES_DB}" AS neg_tables')
cur.execute(f'ATTACH DATABASE "{MATCHED_TR_DB}" AS matched_tr')
cur.execute(f'ATTACH DATABASE "{NEG_TR_DB}" AS neg_tr')

# negated matched transactions
filter_verb_transaction_tables(
    conn=con,
    source_schema='matched_tr',
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    target_schema='neg_tr',
    new_transaction_head='transaction_head',
    new_transaction_row='transaction_row',
    ids_schema='neg_tables',
    ids_table='verb_neg',
    ids_column='id',
    delete_if_exists= True,
    copy_indexes=True,
    verbose= True,
)

CREATE TABLE "neg_tr"."transaction_head" (`id` INTEGER PRIMARY KEY AUTOINCREMENT, `sentence_id` int, `loc` int, `verb` text, `verb_compound` text, `form` text, `deprel` text, `feats` text)
Created table 'neg_tr.transaction_head' (foreign keys ignored).
Copying indexes from 'matched_tr.transaction_head' to 'neg_tr.transaction_head'
CREATE UNIQUE INDEX "neg_tr"."455a687a_0abd3003_transaction_head_uniq" ON "transaction_head"(sentence_id, loc)
Index '455a687a_0abd3003_transaction_head_uniq' created successfully.
CREATE INDEX "neg_tr"."455a687a_0abd3003_`transaction_head_verb`" ON "transaction_head"("`verb`" ASC)
Index '455a687a_0abd3003_`transaction_head_verb`' created successfully.
CREATE INDEX "neg_tr"."455a687a_0abd3003_`transaction_head_verb_compound`" ON "transaction_head"("`verb_compound`" ASC)
Index '455a687a_0abd3003_`transaction_head_verb_compound`' created successfully.
CREATE INDEX "neg_tr"."455a687a_0abd3003_`transaction_head_feats`" ON "transaction_head"("`feats`" ASC)
Index '

True

In [7]:
# non-negated matched transactions
con = sqlite3.connect(VERB_PATTERNS_DB)
cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{MATCHED_TR_DB}" AS matched_tr')
cur.execute(f'ATTACH DATABASE "{NEG_TR_DB}" AS neg_tr')
cur.execute(f'ATTACH DATABASE "{POS_TR_DB}" AS pos_tr')

index_difference(cur,
                 index_tbl_1='matched_tr.transaction_head',
                 id_col_1='id',
                 index_tbl_2='neg_tr.transaction_head',
                 id_col_2='id',
                 output_tbl='pos_tr.index_tbl')

filter_verb_transaction_tables(
    conn=con,
    source_schema='matched_tr',
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    target_schema='pos_tr',
    new_transaction_head='transaction_head',
    new_transaction_row='transaction_row',
    ids_schema='pos_tr',
    ids_table='index_tbl',
    ids_column='idx',
    delete_if_exists= True,
    copy_indexes=True,
    verbose= True,
)

CREATE TABLE "pos_tr"."transaction_head" (`id` INTEGER PRIMARY KEY AUTOINCREMENT, `sentence_id` int, `loc` int, `verb` text, `verb_compound` text, `form` text, `deprel` text, `feats` text)
Created table 'pos_tr.transaction_head' (foreign keys ignored).
Copying indexes from 'matched_tr.transaction_head' to 'pos_tr.transaction_head'
CREATE UNIQUE INDEX "pos_tr"."5e5a01b5_0abd3003_transaction_head_uniq" ON "transaction_head"(sentence_id, loc)
Index '5e5a01b5_0abd3003_transaction_head_uniq' created successfully.
CREATE INDEX "pos_tr"."5e5a01b5_0abd3003_`transaction_head_verb`" ON "transaction_head"("`verb`" ASC)
Index '5e5a01b5_0abd3003_`transaction_head_verb`' created successfully.
CREATE INDEX "pos_tr"."5e5a01b5_0abd3003_`transaction_head_verb_compound`" ON "transaction_head"("`verb_compound`" ASC)
Index '5e5a01b5_0abd3003_`transaction_head_verb_compound`' created successfully.
CREATE INDEX "pos_tr"."5e5a01b5_0abd3003_`transaction_head_feats`" ON "transaction_head"("`feats`" ASC)
Index '

True

## Result

In [8]:
display_sqlite_as_dataframe(MATCHED_TR_DB, 'index_tbl', 10)

,idx
0,4
1,4
2,7
3,16
4,16
5,16
6,16
7,16
8,16
9,20


In [9]:
display_sqlite_as_dataframe(MATCHED_TR_DB, 'transaction_head', 10)

,id,sentence_id,loc,verb,verb_compound,form,deprel,feats
0,4,5,6,oskama,,osanud,acl:relcl,"mod,partic,past,ps"
1,7,7,2,lõpetama,,lõpetas,root,"af,impf,indic,mod,ps,ps3,sg"
2,16,15,11,tulema,,tuli,acl:relcl,"af,impf,indic,mod,ps,ps3,sg"
3,20,18,4,pidama,,peaks,aux,"af,aux,cond,pres,ps"
4,24,20,7,täitma,,täidavad,conj,"af,indic,mod,pl,pres,ps,ps3"
5,25,21,3,muutuma,,muutus,root,"af,aux,impf,indic,ps,ps3,sg"
6,27,22,13,tahtma,,tahtsin,conj,"af,aux,impf,indic,ps,ps1,sg"
7,30,24,2,jõudma,,jõudsin,root,"af,aux,impf,indic,ps,ps1,sg"
8,32,25,5,nägema,,nägin,conj,"af,aux,impf,indic,ps,ps1,sg"
9,34,26,1,mõtlema,,Mõtlesin,root,"af,aux,impf,indic,ps,ps1,sg"


In [10]:
display_sqlite_as_dataframe(MATCHED_TR_DB, 'transaction_row', 10)

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos
0,10,4,5,-1,aux,ei,ei,"aux,neg",NaN,V
1,11,4,7,1,xcomp,laulda,laulma,"aux,inf",NaN,V
2,25,16,9,-2,obl,mille,mis,"gen,pl",NaN,P
3,26,16,10,-1,case,kõrvalt,kõrvalt,post,9.0,K
4,39,24,11,2,obj,tantsupõrandaid,tantsupõrand,"com,part,pl",NaN,S
5,40,25,1,-2,obl,Millest,mis,"el,sg",NaN,P
6,44,27,12,-1,xcomp,lajatada,lajatama,"inf,mod",NaN,V
7,51,30,3,1,obl,tantsumuusikani,tantsumuusika,"com,sg,term",NaN,S
8,52,30,5,2,obl,aastate,aasta,"com,gen,pl",NaN,S
9,53,30,6,3,case,algul,algul,post,5.0,K


In [11]:
display_sqlite_as_dataframe(NEG_TR_DB, 'transaction_head', 10)

,id,sentence_id,loc,verb,verb_compound,form,deprel,feats
0,60,43,12,tahtma,,taha,conj,"imper,main,neg,pres,ps,ps2,sg"
1,63,45,9,õpetama,,õpetanud,conj,"impf,indic,mod,neg,ps"
2,74,50,4,saama,,saa,advcl,"imper,mod,neg,pres,ps,ps2,sg"
3,82,53,8,pidama,,peaks,aux,"aux,cond,neg,pres,ps"
4,101,65,6,oskama,,oska,advcl,"aux,indic,neg,pres,ps"
5,120,75,10,pidama,,peaks,aux,"cond,main,neg,pres,ps"
6,121,75,15,minema,ära,läheks,conj,"aux,cond,neg,pres,ps"
7,149,88,9,saama,,saa,aux,"aux,indic,neg,pres,ps"
8,160,93,11,lootma,,lootnudki,advcl,"impf,indic,mod,neg,ps"
9,172,100,3,lõpetama,,lõpetanud,root,"impf,indic,mod,neg,ps"


In [12]:
display_sqlite_as_dataframe(NEG_TR_DB, 'transaction_row', 10)

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos
0,97,60,11,-1,aux,ei,ei,"aux,neg",None,V
1,98,60,13,1,xcomp,olla,olema,"inf,main",None,V
2,99,60,14,2,xcomp,bussijuhid,bussijuht,"com,nom,pl",None,S
3,102,63,12,1,xcomp,elama,elama,"ill,mod,ps,sup",None,V
4,118,74,3,-1,aux,ei,ei,"aux,neg",None,V
5,119,74,5,1,obl,toidupoest,toidupood,"com,el,sg",None,S
6,120,74,6,2,obj,soovitut,soovitu,"com,part,sg",None,S
7,160,101,5,-1,aux,ei,ei,"aux,neg",None,V
8,161,101,7,1,xcomp,kommenteerida,kommenteerima,"aux,inf",None,V
9,183,121,17,2,obl,käest,käsi,"com,el,sg",None,S


In [13]:
display_sqlite_as_dataframe(POS_TR_DB, 'index_tbl', 10)

,idx
0,4
1,7
2,16
3,20
4,24
5,25
6,27
7,30
8,32
9,34


In [14]:
display_sqlite_as_dataframe(POS_TR_DB, 'transaction_head', 10)

,id,sentence_id,loc,verb,verb_compound,form,deprel,feats
0,4,5,6,oskama,,osanud,acl:relcl,"mod,partic,past,ps"
1,7,7,2,lõpetama,,lõpetas,root,"af,impf,indic,mod,ps,ps3,sg"
2,16,15,11,tulema,,tuli,acl:relcl,"af,impf,indic,mod,ps,ps3,sg"
3,20,18,4,pidama,,peaks,aux,"af,aux,cond,pres,ps"
4,24,20,7,täitma,,täidavad,conj,"af,indic,mod,pl,pres,ps,ps3"
5,25,21,3,muutuma,,muutus,root,"af,aux,impf,indic,ps,ps3,sg"
6,27,22,13,tahtma,,tahtsin,conj,"af,aux,impf,indic,ps,ps1,sg"
7,30,24,2,jõudma,,jõudsin,root,"af,aux,impf,indic,ps,ps1,sg"
8,32,25,5,nägema,,nägin,conj,"af,aux,impf,indic,ps,ps1,sg"
9,34,26,1,mõtlema,,Mõtlesin,root,"af,aux,impf,indic,ps,ps1,sg"


In [15]:
display_sqlite_as_dataframe(POS_TR_DB, 'transaction_row', 10)

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos
0,10,4,5,-1,aux,ei,ei,"aux,neg",NaN,V
1,11,4,7,1,xcomp,laulda,laulma,"aux,inf",NaN,V
2,25,16,9,-2,obl,mille,mis,"gen,pl",NaN,P
3,26,16,10,-1,case,kõrvalt,kõrvalt,post,9.0,K
4,39,24,11,2,obj,tantsupõrandaid,tantsupõrand,"com,part,pl",NaN,S
5,40,25,1,-2,obl,Millest,mis,"el,sg",NaN,P
6,44,27,12,-1,xcomp,lajatada,lajatama,"inf,mod",NaN,V
7,51,30,3,1,obl,tantsumuusikani,tantsumuusika,"com,sg,term",NaN,S
8,52,30,5,2,obl,aastate,aasta,"com,gen,pl",NaN,S
9,53,30,6,3,case,algul,algul,post,5.0,K


NB! Andmebaasi *positive_transactions.db* tabelisse *transaction_row* on mõnel juhul eitust sisaldavad transaktsioonid alles jäänud, kui vastav verb *transaction_head* tabelis ei olnud mingil põhjusel märgitud eitusvormiks.